# 21 - USS-gated region proposal: can the camera segment inside the USS region?

Gate result of notebook 20: the projected USS map object is too coarse for a
POINT prompt (hit 25-34%) but a REGION around it contains the obstacle
(10% of frame -> 74%), and every detector-missed test sequence has a frame
where it does. This notebook asks the next question: given that region, does
SAM3 actually recover the obstacle - class-agnostic, with no object name?

Variants per frame (same frame set for all):
- `box`  : the region as a box prompt to SAM3 (single-frame tracker)
- `text_<p>` : SAM3 text prompt `<p>` ("obstacle", "object"), best mask whose
  box centre lies inside the region
- `nouss_obstacle`: best "obstacle" mask anywhere in the image - the same
  model WITHOUT the USS region (does the region add anything?)

Frames: every corridor frame of the detector-missed test sequences + a matched
control of detected test sequences. GT box = the dataset's pseudo-label (stated
limitation; human click-track gold comes later). Association = one target slot per sequence by its approaching track;
calibration = per (camera, recording day) from same-day TRAIN sequences
(the dev rig was re-mounted between sessions), global fallback.
GPU required (SAM3).

In [ ]:
# --- pipeline bootstrap (identical first cell in every pipeline notebook) ---
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/vision-uss-research")
    REPO_URL = "https://github.com/Ali-RT/vision-uss-research"
    if not (REPO_DIR / ".git").exists():   # fresh VM: clone (no separate bootstrap script)
        subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
    # always run the latest main; discards VM-local edits to tracked repo files
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "-q", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "-q", "--hard", "origin/main"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO_DIR / "requirements" / "colab.txt")], check=True)
    # canmatrix 1.3.0 (2026-10-01) breaks `import asammdf` (TypeError in
    # asammdf/blocks/types.py) - pin until asammdf supports it
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "asammdf==8.8.27",
                    "canmatrix==1.2"], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "pyproject.toml").exists())

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from vision_uss_research.runs import git_sha
from vision_uss_research.settings import load_paths

PROFILE = "colab_drive" if IN_COLAB else "local"
ARTIFACTS_OVERRIDE = None
RAW_ROOT_OVERRIDE = None

paths = load_paths(profile=PROFILE)
ARTIFACTS = Path(ARTIFACTS_OVERRIDE or paths.outputs_dir)
RAW_ROOT = Path(RAW_ROOT_OVERRIDE or paths.raw_data_root)
print(f"repo      : {REPO_DIR}")
print(f"git sha   : {git_sha(REPO_DIR)}")
print(f"artifacts : {ARTIFACTS}")
print(f"raw root  : {RAW_ROOT}")

In [ ]:
import json as _json
import sys

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

sys.path.insert(0, str(REPO_DIR / "scripts"))
import uss_prompt_analysis as upa
from vision_uss_research.method.uss_prompt import (fit_ground_camera, mask_box,
                                                   region_box, track_target_slots)
from vision_uss_research.qa.flags import box_iou
from vision_uss_research.runs import finish_run, new_run_id, start_run
from vision_uss_research.sequences import frames_window_id

DATASET_RUN = "20260718_dataset_v2"
PROJ_RUN = "prompt_uss_proj_v1"           # notebook 20 output
N_CONTROL_SEQS = 150
FRAMES_PER_CONTROL_SEQ = 5
MAX_FRAMES_PER_MISSED_SEQ = 20
REGION = dict(half_w=160, below=100, above=250)
TEXT_PROMPTS = ["obstacle", "object"]
SCORE_THRESHOLD = 0.2
FOUND_IOU = 0.3
SEED = 0

MAN_DIR = ARTIFACTS / "datasets" / DATASET_RUN
PROJ_DIR = ARTIFACTS / "method" / PROJ_RUN
WINDOW_ROOT = ARTIFACTS / "frames" / frames_window_id(20, (0.30, 0.85))
RUN_DIR = ARTIFACTS / "method" / new_run_id("prompt", "region_sam3_v1")
OUT_CSV = RUN_DIR / "region_segmentation.csv"
RUN_DIR.mkdir(parents=True, exist_ok=True)
KEYS = ["sequence_id", "camera", "frame_idx"]

slots, boxes = upa.load(PROJ_DIR, MAN_DIR)
# Association: ONE target slot per (sequence, camera) chosen by its whole
# approaching track (per-frame nearest-in-corridor picks clutter).
tgt = track_target_slots(slots)
pairs = (slots.merge(tgt, on=["sequence_id", "camera", "slot"])
         .merge(boxes, on=KEYS))
pairs["day"] = pairs.sequence_id.str.split("_").str[1]

# Calibration: the dev rig was re-mounted between sessions (one global fit
# leaves ~70-110 px residual, per-day ~33 px), so fit per (camera, recording
# day) on that day's TRAIN-split sequences only - never on the sequence being
# tested - and fall back to one global fit per camera when a day has too few
# pairs. A production camera has a fixed factory calibration; per-session
# calibration is this rig's stand-in for it.
MIN_DAY_PAIRS = 25
glob = {}
for cam_name in ("front", "rear"):
    t = pairs[(pairs.camera == cam_name) & (pairs.split == "train")]
    glob[cam_name], info = fit_ground_camera(t.mid_x, t.mid_y, (t.x0 + t.x1) / 2, t.y1, cam_name)
    print(f"global {cam_name}: median {info['median_px']:.0f}px")
cams, cal_src = {}, {}
for (cam_name, day), t in pairs[pairs.split == "train"].groupby(["camera", "day"]):
    if len(t) >= MIN_DAY_PAIRS:
        cams[(cam_name, day)], _ = fit_ground_camera(t.mid_x, t.mid_y, (t.x0 + t.x1) / 2,
                                                     t.y1, cam_name)
def camera_for(cam_name, day):
    m = cams.get((cam_name, day))
    return (m, "day") if m is not None else (glob[cam_name], "global")
print(f"per-day calibrations: {len(cams)}")
(RUN_DIR / "cameras.json").write_text(_json.dumps(
    {"global": {k: c.as_dict() for k, c in glob.items()},
     "per_day": {f"{k[0]}_{k[1]}": c.as_dict() for k, c in cams.items()}}, indent=1))

detected = set(pd.read_csv(ARTIFACTS / "reports" / "eval_causal" / "frame_predictions.csv",
                           usecols=["sequence_id"]).sequence_id)
test_c = pairs[pairs.split == "test"]
missed = (test_c[~test_c.sequence_id.isin(detected)]
          .groupby("sequence_id").head(MAX_FRAMES_PER_MISSED_SEQ).assign(group="missed"))
ctrl_pool = test_c[test_c.sequence_id.isin(detected)].sequence_id.drop_duplicates()
ctrl_seqs = ctrl_pool.sample(min(N_CONTROL_SEQS, len(ctrl_pool)), random_state=SEED)
control = (test_c[test_c.sequence_id.isin(ctrl_seqs)].sample(frac=1, random_state=SEED)
           .groupby("sequence_id").head(FRAMES_PER_CONTROL_SEQ).assign(group="control"))
todo = pd.concat([missed, control], ignore_index=True)
proj = []
for r in todo.itertuples(index=False):
    m, src = camera_for(r.camera, r.day)
    u, v, ok = m.project(r.mid_x, r.mid_y)
    proj.append((float(u), float(v), bool(ok), src))
todo["pu"], todo["pv"], todo["proj_ok"], todo["calib"] = zip(*proj)
todo = todo[todo.proj_ok]

done = set()
if OUT_CSV.exists():
    prev = pd.read_csv(OUT_CSV)
    done = set(map(tuple, prev[KEYS].astype(str).values))
todo = todo[[tuple(map(str, k)) not in done for k in todo[KEYS].values]]
start_run(RUN_DIR, config={"proj_run": PROJ_RUN, "region": REGION, "text_prompts": TEXT_PROMPTS,
                           "score_threshold": SCORE_THRESHOLD, "found_iou": FOUND_IOU},
          inputs=[f"method/{PROJ_RUN}", f"datasets/{DATASET_RUN}", "reports/eval_causal"],
          repo_root=REPO_DIR)
print(f"calibration: {todo.calib.value_counts().to_dict()}")
print(f"missed seqs {missed.sequence_id.nunique()} ({len(missed)} frames) | "
      f"control seqs {control.sequence_id.nunique()} ({len(control)} frames) | todo {len(todo)}")

## SAM3 inside the USS region

In [ ]:
from vision_uss_research.labeling.sam3 import Sam3Engine
eng = Sam3Engine(score_threshold=SCORE_THRESHOLD)

def in_region(b, reg):
    cx, cy = (b[0] + b[2]) / 2, (b[1] + b[3]) / 2
    return reg[0] <= cx <= reg[2] and reg[1] <= cy <= reg[3]

buf = []
def flush():
    global buf
    if buf:
        pd.DataFrame(buf).to_csv(OUT_CSV, mode="a", header=not OUT_CSV.exists(), index=False)
        buf = []

for r in tqdm(todo.itertuples(index=False), total=len(todo), unit="frame"):
    out = {"sequence_id": r.sequence_id, "camera": r.camera, "frame_idx": r.frame_idx,
           "group": r.group, "bin": r.bin, "class_name": r.class_name,
           "range_mm": r.range_mm, "calib": r.calib, "pu": round(r.pu, 1), "pv": round(r.pv, 1), "error": ""}
    p = WINDOW_ROOT / r.sequence_id / r.camera / f"{int(r.frame_idx):05d}.jpg"
    if not p.exists():
        out["error"] = "no_frame"; buf.append(out); continue
    img = np.array(Image.open(p).convert("RGB"))
    H, W = img.shape[:2]
    gt = (r.x0, r.y0, r.x1, r.y1)
    reg = region_box(r.pu, r.pv, W, H, **REGION)
    out.update(rx0=reg[0], ry0=reg[1], rx1=reg[2], ry1=reg[3],
               gt_in_region=int(in_region(gt, reg)))
    try:
        m = eng.track([Image.fromarray(img)], 0, {"box": reg}).get(0)
        b = mask_box(m) if m is not None else None
        out["box_iou"] = round(box_iou(b, gt), 4) if b else 0.0
        out["box_mask"] = None if b is None else ",".join(f"{v:.0f}" for v in b)
    except Exception as e:
        out["error"] = f"box:{type(e).__name__}"
    for tp in TEXT_PROMPTS:
        try:
            res = eng.segment_text(img, tp)
        except Exception as e:
            out["error"] += f"|text:{type(e).__name__}"; continue
        cand = [(tuple(map(float, bb)), float(sc)) for bb, sc in zip(res["boxes"], res["scores"])]
        inr = [c for c in cand if in_region(c[0], reg)]
        best = max(inr, key=lambda c: c[1]) if inr else None
        out[f"text_{tp}_iou"] = round(box_iou(best[0], gt), 4) if best else 0.0
        out[f"text_{tp}_score"] = round(best[1], 3) if best else None
        if tp == "obstacle":
            anyb = max(cand, key=lambda c: c[1]) if cand else None
            out["nouss_obstacle_iou"] = round(box_iou(anyb[0], gt), 4) if anyb else 0.0
    buf.append(out)
    if len(buf) >= 50:
        flush()
    eng.free()
flush()

## Summary

In [ ]:
res = pd.read_csv(OUT_CSV)
ok = res[res.error.fillna("") == ""]
print(f"frames scored {len(ok)} of {len(res)} | errors {(res.error.fillna('') != '').sum()}")
variants = ["box_iou"] + [f"text_{t}_iou" for t in TEXT_PROMPTS] + ["nouss_obstacle_iou"]
rows = []
for g, d in ok.groupby("group"):
    for v in variants:
        found = d[v] >= FOUND_IOU
        rows.append({"group": g, "variant": v.replace("_iou", ""), "frames": len(d),
                     "gt_in_region": round(d.gt_in_region.mean(), 3),
                     "frame_found": round(found.mean(), 3),
                     "median_iou": round(d[v].median(), 3),
                     "seqs": d.sequence_id.nunique(),
                     "seqs_found": int(d[found].sequence_id.nunique())})
summary = pd.DataFrame(rows)
display(summary)
summary.to_csv(RUN_DIR / "summary.csv", index=False)
finish_run(RUN_DIR, summary={"rows": summary.to_dict(orient="records")})

# gallery: 12 missed-sequence frames - region (yellow), GT (green), box-prompt mask (red)
import matplotlib.pyplot as plt
g = ok[ok.group == "missed"].sort_values("box_iou", ascending=False)
g = pd.concat([g.head(6), g.tail(6)])
fig, axes = plt.subplots(3, 4, figsize=(16, 8))
for ax, r in zip(axes.flat, g.itertuples()):
    im = cv2.cvtColor(cv2.imread(str(WINDOW_ROOT / r.sequence_id / r.camera /
                                     f"{int(r.frame_idx):05d}.jpg")), cv2.COLOR_BGR2RGB)
    b = boxes.set_index(KEYS).loc[(r.sequence_id, r.camera, r.frame_idx)]
    cv2.rectangle(im, (int(r.rx0), int(r.ry0)), (int(r.rx1), int(r.ry1)), (255, 220, 0), 2)
    cv2.rectangle(im, (int(b.x0), int(b.y0)), (int(b.x1), int(b.y1)), (0, 255, 0), 2)
    if isinstance(r.box_mask, str):
        x0, y0, x1, y1 = map(float, r.box_mask.split(","))
        cv2.rectangle(im, (int(x0), int(y0)), (int(x1), int(y1)), (255, 0, 0), 2)
    ax.imshow(im); ax.axis("off")
    ax.set_title(f"{r.class_name} box IoU {r.box_iou:.2f}", fontsize=8)
fig.tight_layout(); fig.savefig(RUN_DIR / "gallery_missed.jpg", dpi=90); plt.show()
print(f"download {RUN_DIR.name}/ and share summary.csv + gallery_missed.jpg")